### This is the "dirty" trial of several models. Tried SARIMA, Prophet, XGBoost, combining their predictions, adding Linear Regression as stacking on top of each combination, adding more features to XGBoost and Linear Regression, Ridge... even tried LSTMs at the end of this notebook.

### For the rigorous trial of the best models of this notebook, please check model_comparison.ipynb, as this notebook is quite long and contains many experimental results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import numpy as np

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.modeling.sarima_model import SARIMAModel
from backend.modeling.prophet_model import ProphetModel
from sklearn.preprocessing import MinMaxScaler
from backend.modeling.xgboost_model import XGBoostModel

In [ ]:
water_engineered_path = PATHS['engineered_data'] / 'water_engineered.parquet'
df = pd.read_parquet(water_engineered_path)
df.head()

### Trying Prophet and SARIMA individually

In [ ]:
reservoir_id = 11
df_res = df[df['id'] == reservoir_id].copy()
df_res = df_res.sort_values('date')

# Set date as index
series = df_res.set_index('date')['storage']

In [ ]:
# Train/test split: last 52 weeks as test
train_series = series.iloc[-(52*8):-52] # 7 years
test_series = series.iloc[-52:] # 1 year

train_stacking_index = train_series[-(52*3):].index  # Last 3 years for training the stacking one
# Create a dataframe with train_index as index
train_stacking = pd.DataFrame(index=train_stacking_index)
train_stacking['storage'] = train_series[-(52*3):]  # Last 3 years from train_series of storage
train_stacking['sarima_prediction'] = np.nan
train_stacking['prophet_prediction'] = np.nan

test_stacking_index = test_series.index  # Last year for testing the stacking one
test_stacking = pd.DataFrame(index=test_stacking_index)
test_stacking['storage'] = test_series
test_stacking['sarima_prediction'] = np.nan
test_stacking['prophet_prediction'] = np.nan

for i in range(3):
    # print(f"Starting SARIMA model training for iteration {i}")
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
    # print(f"\n train_series size: {train_series.shape}")
    sarima_test_pred = sarima_model.predict(steps=52)
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

# Predicting SARIMA for test data
sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
sarima_model.fit(train_series[-(52*4):])
sarima_test_pred = sarima_model.predict(steps=52)
sarima_test_pred.index = test_stacking.index
sarima_test_pred.head()

test_stacking['sarima_prediction'] = sarima_test_pred

for i in range(3):
    # print(f"Starting Prophet model training for iteration {i}")
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
    # print(f"\n train_series size: {train_series.shape}")
    prophet_test_pred = prophet_model.predict(steps=52)
    # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

# Predicting prophet for test data
prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
prophet_model.fit(train_series[-(52*4):])
# print(f"\n Selecting from {train_series.index[-(52*4):]}")
prophet_test_pred = prophet_model.predict(steps=52)
prophet_test_pred.index = test_stacking.index
prophet_test_pred.head()

test_stacking['prophet_prediction'] = prophet_test_pred
# print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
# test_stacking.info()



In [ ]:
plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')


# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

print(train_stacking.tail())

### First version of XGBoost (for definitive versions, please check the backend folder)

In [ ]:
# Create and train XGBoost model
model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

def predict_next_year(X):
    predictions = []
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        X_step['next_storage_value'] = X_step['storage'].shift(-week)
        X_step = X_step.iloc[:-52]
        
        # print(f"\n X_step from step {week}: \n {X_step}")
        model.fit(X_step)
        
        
        # Generate forecasts for 52 steps
        prediction = model.predict(last_row.to_frame().T)
        predictions.append(prediction)

    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

forecast = predict_next_year(train_stacking)

In [ ]:
plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

### Trial of adding data from previous seasons (years) at the same point (week in the year)

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)

# Seasonal lags (quarterly and annual)
for lag in [52, 52*2, 52*3]:
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

def predict_next_year(X):
    predictions = []
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        X_step['next_storage_value'] = X_step['storage'].shift(-week)
        X_step = X_step.iloc[:-52]
        
        # print(f"\n X_step from step {week}: \n {X_step}")
        model.fit(X_step)
        
        
        # Generate forecasts for 52 steps
        prediction = model.predict(last_row.to_frame().T)
        predictions.append(prediction)

    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

# Test the function
forecast = predict_next_year(train)

### Feature engineering for XGBoost

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)
train['capacity'] = train_period['capacity'].astype(float)
train['storage_pct'] = (train['storage'] / train['capacity']).astype(float)

# Temporal features
train['year'] = train_period['year'].astype(float)
train['month'] = train_period['month'].astype(float)
train['week_idx'] = train_period['week_idx'].astype(float)
train['day_of_year'] = train.index.dayofyear.astype(float)
train['is_summer'] = ((train.index.month >= 6) & (train.index.month <= 9)).astype(int)
train['is_winter'] = ((train.index.month >= 12) | (train.index.month <= 2)).astype(int)

# Cyclical encoding for temporal features
train['week_sin'] = np.sin(2 * np.pi * train['week_idx'] / 52).astype(float)
train['week_cos'] = np.cos(2 * np.pi * train['week_idx'] / 52).astype(float)
train['month_sin'] = np.sin(2 * np.pi * (train['month'] - 1) / 12).astype(float)
train['month_cos'] = np.cos(2 * np.pi * (train['month'] - 1) / 12).astype(float)

# Lag features - various time horizons
# Recent lags (1-4 weeks)
for lag in range(1, 5):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Monthly lags (4-12 weeks, every 4 weeks)
for lag in range(4, 13, 4):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Seasonal lags (quarterly and annual)
for lag in [13, 26, 39, 52]:  # Quarter, half-year, 3-quarters, full year
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Rolling statistics
train['storage_mean_4w'] = train['storage'].rolling(window=4, min_periods=1).mean().astype(float)
train['storage_std_4w'] = train['storage'].rolling(window=4, min_periods=1).std().astype(float)
train['storage_mean_12w'] = train['storage'].rolling(window=12, min_periods=1).mean().astype(float)
train['storage_std_12w'] = train['storage'].rolling(window=12, min_periods=1).std().astype(float)
train['storage_mean_26w'] = train['storage'].rolling(window=26, min_periods=1).mean().astype(float)
train['storage_std_26w'] = train['storage'].rolling(window=26, min_periods=1).std().astype(float)

# Rate of change features
train['storage_change_1w'] = train['storage'].diff(1).astype(float)
train['storage_change_4w'] = train['storage'].diff(4).astype(float)
train['storage_change_12w'] = train['storage'].diff(12).astype(float)
train['storage_pct_change_1w'] = train['storage_pct'].diff(1).astype(float)
train['storage_pct_change_4w'] = train['storage_pct'].diff(4).astype(float)

# Trend features
train['storage_trend_4w'] = train['storage'].rolling(window=4).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 4 else np.nan, raw=False).astype(float)
train['storage_trend_12w'] = train['storage'].rolling(window=12).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 12 else np.nan, raw=False).astype(float)

# Min/Max features over different windows
train['storage_min_4w'] = train['storage'].rolling(window=4, min_periods=1).min().astype(float)
train['storage_max_4w'] = train['storage'].rolling(window=4, min_periods=1).max().astype(float)
train['storage_min_12w'] = train['storage'].rolling(window=12, min_periods=1).min().astype(float)
train['storage_max_12w'] = train['storage'].rolling(window=12, min_periods=1).max().astype(float)
train['storage_min_26w'] = train['storage'].rolling(window=26, min_periods=1).min().astype(float)
train['storage_max_26w'] = train['storage'].rolling(window=26, min_periods=1).max().astype(float)

# Relative position features
train['storage_vs_4w_mean'] = (train['storage'] - train['storage_mean_4w']).astype(float)
train['storage_vs_12w_mean'] = (train['storage'] - train['storage_mean_12w']).astype(float)
train['storage_vs_26w_mean'] = (train['storage'] - train['storage_mean_26w']).astype(float)

# Base model predictions as features
train['sarima_prediction'] = train_stacking['sarima_prediction'].astype(float)
train['prophet_prediction'] = train_stacking['prophet_prediction'].astype(float)

# Volatility features
train['storage_volatility_4w'] = (train['storage'].rolling(window=4).std() / train['storage'].rolling(window=4).mean()).astype(float)
train['storage_volatility_12w'] = (train['storage'].rolling(window=12).std() / train['storage'].rolling(window=12).mean()).astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Final check - ensure all columns are numeric
print(f"Data types after conversion:")
print(train.dtypes.value_counts())
print(f"\nCreated comprehensive train dataset with shape: {train.shape}")
print(f"Features: {train.shape[1]}")
print(f"Index matches train_stacking: {train.index.equals(train_stacking.index)}")

# Updated predict_next_year function to work with the new train dataset
def predict_next_year(X):
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)
    predictions = []
    
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        
        # Create target variable: storage value 'week' weeks ahead
        X_step['next_storage_value'] = X_step['storage'].shift(-week).astype(float)
        
        # Remove rows where target is NaN (last 'week' rows)
        X_step = X_step.iloc[:-week].dropna()
        
        if len(X_step) > 0:
            # Ensure all data is float type for XGBoost
            X_step = X_step.astype(float)
            last_row_df = last_row.to_frame().T.astype(float)
            
            # Fit model on available data
            model.fit(X_step)
            
            # Predict using the last available row
            prediction = model.predict(last_row_df)
            predictions.append(prediction[0])
        else:
            # Fallback if no data available
            predictions.append(float(last_row['storage']))
    
    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

print("\nDataset ready for prediction!")
print("Call: forecast = predict_next_year(train)")

# Display first few rows to check
print(f"\nFirst 5 rows of train dataset:")
print(train.head())

# Test the function
forecast = predict_next_year(train)

In [ ]:
plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

### Feature engineering for XGBoost, modifying several features between trials

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)
train['capacity'] = train_period['capacity'].astype(float)
train['storage_pct'] = (train['storage'] / train['capacity']).astype(float)

# Temporal features
train['year'] = train_period['year'].astype(float)
train['month'] = train_period['month'].astype(float)
train['week_idx'] = train_period['week_idx'].astype(float)
train['is_summer'] = ((train.index.month >= 6) & (train.index.month <= 9)).astype(int)
train['is_winter'] = ((train.index.month >= 12) | (train.index.month <= 2)).astype(int)

# Cyclical encoding for temporal features
train['week_sin'] = np.sin(2 * np.pi * train['week_idx'] / 52).astype(float)
train['week_cos'] = np.cos(2 * np.pi * train['week_idx'] / 52).astype(float)
train['month_sin'] = np.sin(2 * np.pi * (train['month'] - 1) / 12).astype(float)
train['month_cos'] = np.cos(2 * np.pi * (train['month'] - 1) / 12).astype(float)

# Lag features - various time horizons
for lag in range(1, 9):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

for lag in range(52-5, 52+6):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

for lag in range(52*2-5, 52*2+6):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Rolling statistics
train['storage_mean_4w'] = train['storage'].rolling(window=4, min_periods=1).mean().astype(float)
train['storage_std_4w'] = train['storage'].rolling(window=4, min_periods=1).std().astype(float)

# Rate of change features
train['storage_change_1w'] = train['storage'].diff(1).astype(float)
train['storage_change_4w'] = train['storage'].diff(4).astype(float)
train['storage_change_12w'] = train['storage'].diff(12).astype(float)
train['storage_pct_change_1w'] = train['storage_pct'].diff(1).astype(float)
train['storage_pct_change_4w'] = train['storage_pct'].diff(4).astype(float)

# Trend features
train['storage_trend_4w'] = train['storage'].rolling(window=4).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 4 else np.nan, raw=False).astype(float)
train['storage_trend_12w'] = train['storage'].rolling(window=12).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 12 else np.nan, raw=False).astype(float)

# Min/Max features over different windows
train['storage_min_4w'] = train['storage'].rolling(window=4, min_periods=1).min().astype(float)
train['storage_max_4w'] = train['storage'].rolling(window=4, min_periods=1).max().astype(float)
train['storage_min_12w'] = train['storage'].rolling(window=12, min_periods=1).min().astype(float)
train['storage_max_12w'] = train['storage'].rolling(window=12, min_periods=1).max().astype(float)
train['storage_min_26w'] = train['storage'].rolling(window=26, min_periods=1).min().astype(float)
train['storage_max_26w'] = train['storage'].rolling(window=26, min_periods=1).max().astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Updated predict_next_year function to work with the new train dataset
def predict_next_year(X):
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)
    predictions = []
    
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        
        # Create target variable: storage value 'week' weeks ahead
        X_step['next_storage_value'] = X_step['storage'].shift(-week).astype(float)
        
        # Remove rows where target is NaN (last 'week' rows)
        X_step = X_step.iloc[:-week].dropna()
        
        if len(X_step) > 0:
            # Ensure all data is float type for XGBoost
            X_step = X_step.astype(float)
            last_row_df = last_row.to_frame().T.astype(float)
            
            # Fit model on available data
            model.fit(X_step)
            
            # Predict using the last available row
            prediction = model.predict(last_row_df)
            predictions.append(prediction[0])
        else:
            # Fallback if no data available
            predictions.append(float(last_row['storage']))
    
    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

# Test the function
forecast = predict_next_year(train)

plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)
train['capacity'] = train_period['capacity'].astype(float)
train['storage_pct'] = (train['storage'] / train['capacity']).astype(float)

# Temporal features
train['year'] = train_period['year'].astype(float)
train['month'] = train_period['month'].astype(float)
train['week_idx'] = train_period['week_idx'].astype(float)

# Lag features - various time horizons
for lag in range(1, 9):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

for lag in range(52-5, 52+6):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

for lag in range(52*2-5, 52*2+6):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Rolling statistics
train['storage_mean_4w'] = train['storage'].rolling(window=4, min_periods=1).mean().astype(float)
train['storage_std_4w'] = train['storage'].rolling(window=4, min_periods=1).std().astype(float)

# Rate of change features
train['storage_change_1w'] = train['storage'].diff(1).astype(float)
train['storage_change_4w'] = train['storage'].diff(4).astype(float)
train['storage_change_12w'] = train['storage'].diff(12).astype(float)
train['storage_pct_change_1w'] = train['storage_pct'].diff(1).astype(float)
train['storage_pct_change_4w'] = train['storage_pct'].diff(4).astype(float)

# Trend features
train['storage_trend_4w'] = train['storage'].rolling(window=4).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 4 else np.nan, raw=False).astype(float)
train['storage_trend_12w'] = train['storage'].rolling(window=12).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 12 else np.nan, raw=False).astype(float)

# Min/Max features over different windows
train['storage_min_4w'] = train['storage'].rolling(window=4, min_periods=1).min().astype(float)
train['storage_max_4w'] = train['storage'].rolling(window=4, min_periods=1).max().astype(float)
train['storage_min_12w'] = train['storage'].rolling(window=12, min_periods=1).min().astype(float)
train['storage_max_12w'] = train['storage'].rolling(window=12, min_periods=1).max().astype(float)
train['storage_min_26w'] = train['storage'].rolling(window=26, min_periods=1).min().astype(float)
train['storage_max_26w'] = train['storage'].rolling(window=26, min_periods=1).max().astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Updated predict_next_year function to work with the new train dataset
def predict_next_year(X):
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)
    predictions = []
    
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        
        # Create target variable: storage value 'week' weeks ahead
        X_step['next_storage_value'] = X_step['storage'].shift(-week).astype(float)
        
        # Remove rows where target is NaN (last 'week' rows)
        X_step = X_step.iloc[:-week].dropna()
        
        if len(X_step) > 0:
            # Ensure all data is float type for XGBoost
            X_step = X_step.astype(float)
            last_row_df = last_row.to_frame().T.astype(float)
            
            # Fit model on available data
            model.fit(X_step)
            
            # Predict using the last available row
            prediction = model.predict(last_row_df)
            predictions.append(prediction[0])
        else:
            # Fallback if no data available
            predictions.append(float(last_row['storage']))
    
    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

# Test the function
forecast = predict_next_year(train)

plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)
train['capacity'] = train_period['capacity'].astype(float)
train['storage_pct'] = (train['storage'] / train['capacity']).astype(float)


# Rolling statistics
train['storage_mean_4w'] = train['storage'].rolling(window=4, min_periods=1).mean().astype(float)
train['storage_std_4w'] = train['storage'].rolling(window=4, min_periods=1).std().astype(float)

# Rate of change features
train['storage_change_1w'] = train['storage'].diff(1).astype(float)
train['storage_change_4w'] = train['storage'].diff(4).astype(float)
train['storage_change_12w'] = train['storage'].diff(12).astype(float)
train['storage_pct_change_1w'] = train['storage_pct'].diff(1).astype(float)
train['storage_pct_change_4w'] = train['storage_pct'].diff(4).astype(float)

# Trend features
train['storage_trend_4w'] = train['storage'].rolling(window=4).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 4 else np.nan, raw=False).astype(float)
train['storage_trend_12w'] = train['storage'].rolling(window=12).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 12 else np.nan, raw=False).astype(float)

# Min/Max features over different windows
train['storage_min_4w'] = train['storage'].rolling(window=4, min_periods=1).min().astype(float)
train['storage_max_4w'] = train['storage'].rolling(window=4, min_periods=1).max().astype(float)
train['storage_min_12w'] = train['storage'].rolling(window=12, min_periods=1).min().astype(float)
train['storage_max_12w'] = train['storage'].rolling(window=12, min_periods=1).max().astype(float)
train['storage_min_26w'] = train['storage'].rolling(window=26, min_periods=1).min().astype(float)
train['storage_max_26w'] = train['storage'].rolling(window=26, min_periods=1).max().astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Updated predict_next_year function to work with the new train dataset
def predict_next_year(X):
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)
    predictions = []
    
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        
        # Create target variable: storage value 'week' weeks ahead
        X_step['next_storage_value'] = X_step['storage'].shift(-week).astype(float)
        
        # Remove rows where target is NaN (last 'week' rows)
        X_step = X_step.iloc[:-week].dropna()
        
        if len(X_step) > 0:
            # Ensure all data is float type for XGBoost
            X_step = X_step.astype(float)
            last_row_df = last_row.to_frame().T.astype(float)
            
            # Fit model on available data
            model.fit(X_step)
            
            # Predict using the last available row
            prediction = model.predict(last_row_df)
            predictions.append(prediction[0])
        else:
            # Fallback if no data available
            predictions.append(float(last_row['storage']))
    
    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

# Test the function
forecast = predict_next_year(train)

plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

In [ ]:
# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)
train['capacity'] = train_period['capacity'].astype(float)
train['storage_pct'] = (train['storage'] / train['capacity']).astype(float)

# Temporal features
train['year'] = train_period['year'].astype(float)
train['month'] = train_period['month'].astype(float)
train['week_idx'] = train_period['week_idx'].astype(float)

# Cyclical encoding for temporal features
train['week_sin'] = np.sin(2 * np.pi * train['week_idx'] / 52).astype(float)
train['week_cos'] = np.cos(2 * np.pi * train['week_idx'] / 52).astype(float)
train['month_sin'] = np.sin(2 * np.pi * (train['month'] - 1) / 12).astype(float)
train['month_cos'] = np.cos(2 * np.pi * (train['month'] - 1) / 12).astype(float)

# Lag features - various time horizons
# Recent lags (1-4 weeks)
for lag in range(1, 5):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Monthly lags (4-12 weeks, every 4 weeks)
for lag in range(4, 13, 4):
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Seasonal lags (quarterly and annual)
for lag in [13, 26, 39, 52]:  # Quarter, half-year, 3-quarters, full year
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
    train[f'storage_pct_lag_{lag}w'] = train['storage_pct'].shift(lag).astype(float)

# Rolling statistics
train['storage_mean_4w'] = train['storage'].rolling(window=4, min_periods=1).mean().astype(float)
train['storage_std_4w'] = train['storage'].rolling(window=4, min_periods=1).std().astype(float)
train['storage_mean_12w'] = train['storage'].rolling(window=12, min_periods=1).mean().astype(float)
train['storage_std_12w'] = train['storage'].rolling(window=12, min_periods=1).std().astype(float)

# Trend features
train['storage_trend_4w'] = train['storage'].rolling(window=4).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 4 else np.nan, raw=False).astype(float)
train['storage_trend_12w'] = train['storage'].rolling(window=12).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 12 else np.nan, raw=False).astype(float)

# Min/Max features over different windows
train['storage_min_4w'] = train['storage'].rolling(window=4, min_periods=1).min().astype(float)
train['storage_max_4w'] = train['storage'].rolling(window=4, min_periods=1).max().astype(float)
train['storage_min_12w'] = train['storage'].rolling(window=12, min_periods=1).min().astype(float)
train['storage_max_12w'] = train['storage'].rolling(window=12, min_periods=1).max().astype(float)

# Relative position features
train['storage_vs_4w_mean'] = (train['storage'] - train['storage_mean_4w']).astype(float)
train['storage_vs_12w_mean'] = (train['storage'] - train['storage_mean_12w']).astype(float)


# Base model predictions as features
train['sarima_prediction'] = train_stacking['sarima_prediction'].astype(float)
train['prophet_prediction'] = train_stacking['prophet_prediction'].astype(float)

# Volatility features
train['storage_volatility_4w'] = (train['storage'].rolling(window=4).std() / train['storage'].rolling(window=4).mean()).astype(float)
train['storage_volatility_12w'] = (train['storage'].rolling(window=12).std() / train['storage'].rolling(window=12).mean()).astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Updated predict_next_year function to work with the new train dataset
def predict_next_year(X):
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)
    predictions = []
    
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        
        # Create target variable: storage value 'week' weeks ahead
        X_step['next_storage_value'] = X_step['storage'].shift(-week).astype(float)
        
        # Remove rows where target is NaN (last 'week' rows)
        X_step = X_step.iloc[:-week].dropna()
        
        if len(X_step) > 0:
            # Ensure all data is float type for XGBoost
            X_step = X_step.astype(float)
            last_row_df = last_row.to_frame().T.astype(float)
            
            # Fit model on available data
            model.fit(X_step)
            
            # Predict using the last available row
            prediction = model.predict(last_row_df)
            predictions.append(prediction[0])
        else:
            # Fallback if no data available
            predictions.append(float(last_row['storage']))
    
    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast


# Test the function
forecast = predict_next_year(train)


plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

In [ ]:
reservoir_id = 11
df_res = df[df['id'] == reservoir_id].copy()
df_res = df_res.sort_values('date')

In [ ]:
# Set date as index
series_xgboost = df_res.set_index('date')
series_xgboost.drop(columns=['id', 'province', 'day', 'autonomous_community'], inplace=True)
series_xgboost.loc[:, 'week_idx'] = series_xgboost['week_idx'].astype(float)
series_xgboost.loc[:, 'storage'] = series_xgboost['storage'].astype(float)

# Train/test split: last 52 weeks as test
train_series_xgboost = series_xgboost.iloc[-(52*8):-52] # 7 years
test_series_xgboost = series_xgboost.iloc[-52:] # 1 year

# Create a dataframe with train_index as index - FIX: Create proper copy
train_stacking_xgboost = train_series_xgboost[-(52*3):].copy()  # Last 3 years from train_series of storage
train_stacking_xgboost['xgboost_prediction'] = np.nan

# Create a dataframe with test_index as index - FIX: Create proper copy
test_stacking_xgboost = test_series_xgboost.copy()
test_stacking_xgboost['xgboost_prediction'] = np.nan

# Create and train XGBoost model
model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

def predict_next_year(X):
    predictions = []  # Reset predictions for each call
    for week in range(1, 53):
        X_step = X.copy()
        last_row = X_step.iloc[-1]
        X_step.loc[:, 'next_storage_value'] = X_step['storage'].shift(-week)
        X_step = X_step.iloc[:-52]
        
        # print(f"\n X_step from step {week}: \n {X_step}")
        model.fit(X_step)
        
        # Generate forecasts for 52 steps - FIX: Extract scalar value
        prediction = model.predict(last_row.to_frame().T)
        predictions.append(float(prediction[0]))  # Convert to scalar float

    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking_xgboost.index, dtype='float64')
    return forecast

for i in range(3):
    xgboost_test_pred = predict_next_year(train_stacking_xgboost)
    # FIX: Convert to proper dtype before assignment
    train_stacking_xgboost.iloc[(52*i):(52*(i+1)), train_stacking_xgboost.columns.get_loc('xgboost_prediction')] = xgboost_test_pred.values.astype('float64')

xgboost_test_pred = predict_next_year(train_series_xgboost[-(52*4):])
xgboost_test_pred.index = test_stacking_xgboost.index

test_stacking_xgboost['xgboost_prediction'] = xgboost_test_pred.astype('float64')

In [ ]:
plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(train_stacking.index, train_stacking_xgboost['xgboost_prediction'], label='XGBoost_train', color='purple', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, test_stacking_xgboost['xgboost_prediction'], label='XGBoost_test', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

### First contact with Linear Regression

In [ ]:
# Train Linear Regression Stacking Model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare features (predictions from base models) and target (actual storage)
X_train = train[['sarima', 'prophet', 'xgboost']].copy()
y_train = train['storage'].copy()

X_test = test[['sarima', 'prophet', 'xgboost']].copy()
y_test = test['storage'].copy()

# Remove any rows with NaN values
train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]
X_test_clean = X_test[test_mask]
y_test_clean = y_test[test_mask]

# Train the linear regression stacking model
stacking_model = LinearRegression()
stacking_model.fit(X_train_clean, y_train_clean)

# Make predictions
train_stacking_pred = stacking_model.predict(X_train_clean)
test_stacking_pred = stacking_model.predict(X_test_clean)

# Create prediction series with proper indices
train_stacking_predictions = pd.Series(train_stacking_pred, index=X_train_clean.index, name='stacking_prediction')
test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

# Calculate performance metrics
train_mse = mean_squared_error(y_train_clean, train_stacking_pred)
train_mae = mean_absolute_error(y_train_clean, train_stacking_pred)
train_r2 = r2_score(y_train_clean, train_stacking_pred)

test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
test_r2 = r2_score(y_test_clean, test_stacking_pred)

print("Linear Regression Stacking Model Performance:")
print(f"Train - MSE: {train_mse:.4f}, MAE: {train_mae:.4f}, R²: {train_r2:.4f}")
print(f"Test  - MSE: {test_mse:.4f}, MAE: {test_mae:.4f}, R²: {test_r2:.4f}")

print(f"\nModel Coefficients:")
print(f"SARIMA: {stacking_model.coef_[0]:.4f}")
print(f"Prophet: {stacking_model.coef_[1]:.4f}")
print(f"XGBoost: {stacking_model.coef_[2]:.4f}")
print(f"Intercept: {stacking_model.intercept_:.4f}")

# Create comprehensive plot
plt.figure(figsize=(15, 8))

# Plot training data and predictions
plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
plt.plot(train.index, train['xgboost'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)
plt.plot(train_stacking_predictions.index, train_stacking_predictions, label='Stacking Train', color='orange', linestyle='-', linewidth=2)

# Plot test data and predictions
plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
plt.plot(test.index, test['xgboost'], label='XGBoost Test', color='purple', alpha=0.8)
plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

# Add train/test split line
plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

# Formatting
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
plt.xlabel('Date')
plt.ylabel('Storage')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Display model weights interpretation
print(f"\nModel Interpretation:")
total_weight = sum(abs(coef) for coef in stacking_model.coef_)
print(f"Relative importance:")
print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")
print(f"XGBoost: {abs(stacking_model.coef_[2])/total_weight*100:.1f}%")

### First workflow using stacking: SARIMA, Prophet, XGBoost, and Linear Regression as Stacking

In [ ]:
def workflow_for_reservoir(reservoir_id):
    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    # Set date as index
    series = df_res.set_index('date')['storage']

    # Train/test split: last 52 weeks as test
    train_series = series.iloc[-(52*8):-52] # 7 years
    test_series = series.iloc[-52:] # 1 year

    train_stacking_index = train_series[-(52*3):].index  # Last 3 years for training the stacking one
    # Create a dataframe with train_index as index
    train_stacking = pd.DataFrame(index=train_stacking_index)
    train_stacking['storage'] = train_series[-(52*3):]  # Last 3 years from train_series of storage
    train_stacking['sarima_prediction'] = np.nan
    train_stacking['prophet_prediction'] = np.nan

    test_stacking_index = test_series.index  # Last year for testing the stacking one
    test_stacking = pd.DataFrame(index=test_stacking_index)
    test_stacking['storage'] = test_series
    test_stacking['sarima_prediction'] = np.nan
    test_stacking['prophet_prediction'] = np.nan

    for i in range(3):
        # print(f"Starting SARIMA model training for iteration {i}")
        sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
        sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
        # print(f"\n train_series size: {train_series.shape}")
        sarima_test_pred = sarima_model.predict(steps=52)
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

    # Predicting SARIMA for test data
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series[-(52*4):])
    sarima_test_pred = sarima_model.predict(steps=52)
    sarima_test_pred.index = test_stacking.index
    sarima_test_pred.head()

    test_stacking['sarima_prediction'] = sarima_test_pred

    for i in range(3):
        # print(f"Starting Prophet model training for iteration {i}")
        prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
        prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
        # print(f"\n train_series size: {train_series.shape}")
        prophet_test_pred = prophet_model.predict(steps=52)
        # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

    # Predicting prophet for test data
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series[-(52*4):])
    # print(f"\n Selecting from {train_series.index[-(52*4):]}")
    prophet_test_pred = prophet_model.predict(steps=52)
    prophet_test_pred.index = test_stacking.index
    prophet_test_pred.head()

    test_stacking['prophet_prediction'] = prophet_test_pred
    # print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
    # test_stacking.info()

    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    # Set date as index
    series_xgboost = df_res.set_index('date')
    series_xgboost.drop(columns=['id', 'province', 'day', 'autonomous_community'], inplace=True)
    series_xgboost.loc[:, 'week_idx'] = series_xgboost['week_idx'].astype(float)
    series_xgboost.loc[:, 'storage'] = series_xgboost['storage'].astype(float)

    # Train/test split: last 52 weeks as test
    train_series_xgboost = series_xgboost.iloc[-(52*8):-52] # 7 years
    test_series_xgboost = series_xgboost.iloc[-52:] # 1 year

    # Create a dataframe with train_index as index - FIX: Create proper copy
    train_stacking_xgboost = train_series_xgboost[-(52*3):].copy()  # Last 3 years from train_series of storage
    train_stacking_xgboost['xgboost_prediction'] = np.nan

    # Create a dataframe with test_index as index - FIX: Create proper copy
    test_stacking_xgboost = test_series_xgboost.copy()
    test_stacking_xgboost['xgboost_prediction'] = np.nan

    # Create and train XGBoost model
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

    def predict_next_year(X):
        predictions = []  # Reset predictions for each call
        for week in range(1, 53):
            X_step = X.copy()
            last_row = X_step.iloc[-1]
            X_step.loc[:, 'next_storage_value'] = X_step['storage'].shift(-week)
            X_step = X_step.iloc[:-52]
            
            # print(f"\n X_step from step {week}: \n {X_step}")
            model.fit(X_step)
            
            # Generate forecasts for 52 steps - FIX: Extract scalar value
            prediction = model.predict(last_row.to_frame().T)
            predictions.append(float(prediction[0]))  # Convert to scalar float

        # Create forecast series with same index as test_stacking
        forecast = pd.Series(predictions, index=test_stacking_xgboost.index, dtype='float64')
        return forecast

    for i in range(3):
        xgboost_test_pred = predict_next_year(train_stacking_xgboost)
        # FIX: Convert to proper dtype before assignment
        train_stacking_xgboost.iloc[(52*i):(52*(i+1)), train_stacking_xgboost.columns.get_loc('xgboost_prediction')] = xgboost_test_pred.values.astype('float64')

    xgboost_test_pred = predict_next_year(train_series_xgboost[-(52*4):])
    xgboost_test_pred.index = test_stacking_xgboost.index

    test_stacking_xgboost['xgboost_prediction'] = xgboost_test_pred.astype('float64')


    train = pd.DataFrame(index=train_stacking.index)
    test = pd.DataFrame(index=test_stacking.index)

    train['storage'] = train_stacking['storage'].astype(float)
    train['sarima'] = train_stacking['sarima_prediction'].astype(float)
    train['prophet'] = train_stacking['prophet_prediction'].astype(float)
    train['xgboost'] = train_stacking_xgboost['xgboost_prediction'].astype(float)

    test['storage'] = test_stacking['storage'].astype(float)
    test['sarima'] = test_stacking['sarima_prediction'].astype(float)
    test['prophet'] = test_stacking['prophet_prediction'].astype(float)
    test['xgboost'] = test_stacking_xgboost['xgboost_prediction'].astype(float)

    # Train Linear Regression Stacking Model
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Prepare features (predictions from base models) and target (actual storage)
    X_train = train[['sarima', 'prophet', 'xgboost']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima', 'prophet', 'xgboost']].copy()
    y_test = test['storage'].copy()

    # Remove any rows with NaN values
    train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
    test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]

    # Train the linear regression stacking model
    stacking_model = LinearRegression()
    stacking_model.fit(X_train_clean, y_train_clean)

    # Make predictions
    train_stacking_pred = stacking_model.predict(X_train_clean)
    test_stacking_pred = stacking_model.predict(X_test_clean)

    # Create prediction series with proper indices
    train_stacking_predictions = pd.Series(train_stacking_pred, index=X_train_clean.index, name='stacking_prediction')
    test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

    # Calculate performance metrics
    train_mse = mean_squared_error(y_train_clean, train_stacking_pred)
    train_mae = mean_absolute_error(y_train_clean, train_stacking_pred)
    train_r2 = r2_score(y_train_clean, train_stacking_pred)

    test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
    test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
    test_r2 = r2_score(y_test_clean, test_stacking_pred)

    print("Linear Regression Stacking Model Performance:")
    print(f"Train - MSE: {train_mse:.4f}, MAE: {train_mae:.4f}, R²: {train_r2:.4f}")
    print(f"Test  - MSE: {test_mse:.4f}, MAE: {test_mae:.4f}, R²: {test_r2:.4f}")

    print(f"\nModel Coefficients:")
    print(f"SARIMA: {stacking_model.coef_[0]:.4f}")
    print(f"Prophet: {stacking_model.coef_[1]:.4f}")
    print(f"XGBoost: {stacking_model.coef_[2]:.4f}")
    print(f"Intercept: {stacking_model.intercept_:.4f}")

    # Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['xgboost'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)
    plt.plot(train_stacking_predictions.index, train_stacking_predictions, label='Stacking Train', color='orange', linestyle='-', linewidth=2)

    # Plot test data and predictions
    plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(test.index, test['xgboost'], label='XGBoost Test', color='purple', alpha=0.8)
    plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

    # Add train/test split line
    plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Display model weights interpretation
    print(f"\nModel Interpretation:")
    total_weight = sum(abs(coef) for coef in stacking_model.coef_)
    print(f"Relative importance:")
    print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
    print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")
    print(f"XGBoost: {abs(stacking_model.coef_[2])/total_weight*100:.1f}%")

### Several visualizations to start to compare results

In [ ]:
workflow_for_reservoir(9)

In [ ]:
workflow_for_reservoir(5)

### Workflow using only SARIMA and Prophet

In [ ]:
def workflow_for_reservoir_sarima_and_prophet(reservoir_id):
    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    # Set date as index
    series = df_res.set_index('date')['storage']

    # Train/test split: last 52 weeks as test
    train_series = series.iloc[-(52*8):-52] # 7 years
    test_series = series.iloc[-52:] # 1 year

    train_stacking_index = train_series[-(52*3):].index  # Last 3 years for training the stacking one
    # Create a dataframe with train_index as index
    train_stacking = pd.DataFrame(index=train_stacking_index)
    train_stacking['storage'] = train_series[-(52*3):]  # Last 3 years from train_series of storage
    train_stacking['sarima_prediction'] = np.nan
    train_stacking['prophet_prediction'] = np.nan

    test_stacking_index = test_series.index  # Last year for testing the stacking one
    test_stacking = pd.DataFrame(index=test_stacking_index)
    test_stacking['storage'] = test_series
    test_stacking['sarima_prediction'] = np.nan
    test_stacking['prophet_prediction'] = np.nan

    for i in range(3):
        # print(f"Starting SARIMA model training for iteration {i}")
        sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
        sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
        # print(f"\n train_series size: {train_series.shape}")
        sarima_test_pred = sarima_model.predict(steps=52)
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

    # Predicting SARIMA for test data
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series[-(52*4):])
    sarima_test_pred = sarima_model.predict(steps=52)
    sarima_test_pred.index = test_stacking.index
    sarima_test_pred.head()

    test_stacking['sarima_prediction'] = sarima_test_pred

    for i in range(3):
        # print(f"Starting Prophet model training for iteration {i}")
        prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
        prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
        # print(f"\n train_series size: {train_series.shape}")
        prophet_test_pred = prophet_model.predict(steps=52)
        # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

    # Predicting prophet for test data
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series[-(52*4):])
    # print(f"\n Selecting from {train_series.index[-(52*4):]}")
    prophet_test_pred = prophet_model.predict(steps=52)
    prophet_test_pred.index = test_stacking.index
    prophet_test_pred.head()

    test_stacking['prophet_prediction'] = prophet_test_pred
    # print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
    # test_stacking.info()

    train = pd.DataFrame(index=train_stacking.index)
    test = pd.DataFrame(index=test_stacking.index)

    train['storage'] = train_stacking['storage'].astype(float)
    train['sarima'] = train_stacking['sarima_prediction'].astype(float)
    train['prophet'] = train_stacking['prophet_prediction'].astype(float)

    test['storage'] = test_stacking['storage'].astype(float)
    test['sarima'] = test_stacking['sarima_prediction'].astype(float)
    test['prophet'] = test_stacking['prophet_prediction'].astype(float)

    # Train Linear Regression Stacking Model
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Prepare features (predictions from base models) and target (actual storage)
    X_train = train[['sarima', 'prophet']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima', 'prophet']].copy()
    y_test = test['storage'].copy()

    # Remove any rows with NaN values
    train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
    test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]

    # Train the linear regression stacking model
    stacking_model = LinearRegression()
    stacking_model.fit(X_train_clean, y_train_clean)

    # Make predictions
    train_stacking_pred = stacking_model.predict(X_train_clean)
    test_stacking_pred = stacking_model.predict(X_test_clean)

    # Create prediction series with proper indices
    train_stacking_predictions = pd.Series(train_stacking_pred, index=X_train_clean.index, name='stacking_prediction')
    test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

    # Calculate performance metrics
    train_mse = mean_squared_error(y_train_clean, train_stacking_pred)
    train_mae = mean_absolute_error(y_train_clean, train_stacking_pred)
    train_r2 = r2_score(y_train_clean, train_stacking_pred)

    test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
    test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
    test_r2 = r2_score(y_test_clean, test_stacking_pred)

    print("Linear Regression Stacking Model Performance:")
    print(f"Train - MSE: {train_mse:.4f}, MAE: {train_mae:.4f}, R²: {train_r2:.4f}")
    print(f"Test  - MSE: {test_mse:.4f}, MAE: {test_mae:.4f}, R²: {test_r2:.4f}")

    print(f"\nModel Coefficients:")
    print(f"SARIMA: {stacking_model.coef_[0]:.4f}")
    print(f"Prophet: {stacking_model.coef_[1]:.4f}")
    print(f"Intercept: {stacking_model.intercept_:.4f}")

    # Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train_stacking_predictions.index, train_stacking_predictions, label='Stacking Train', color='orange', linestyle='-', linewidth=2)

    # Plot test data and predictions
    plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

    # Add train/test split line
    plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Display model weights interpretation
    print(f"\nModel Interpretation:")
    total_weight = sum(abs(coef) for coef in stacking_model.coef_)
    print(f"Relative importance:")
    print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
    print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")

### More comparisons between models

In [ ]:
workflow_for_reservoir_sarima_and_prophet(11)

In [ ]:
workflow_for_reservoir_sarima_and_prophet(9)

In [ ]:
workflow_for_reservoir_sarima_and_prophet(19)

### Trying a ponderation as stacking instead of user Linear Regression

In [ ]:
def ponderation_stacking(reservoir_id):
    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    # Set date as index
    series = df_res.set_index('date')['storage']

    # Train/test split: last 52 weeks as test
    train_series = series.iloc[-(52*8):-52] # 7 years
    test_series = series.iloc[-52:] # 1 year

    train_stacking_index = train_series[-(52*3):].index  # Last 3 years for training the stacking one
    # Create a dataframe with train_index as index
    train_stacking = pd.DataFrame(index=train_stacking_index)
    train_stacking['storage'] = train_series[-(52*3):]  # Last 3 years from train_series of storage
    train_stacking['sarima_prediction'] = np.nan
    train_stacking['prophet_prediction'] = np.nan

    test_stacking_index = test_series.index  # Last year for testing the stacking one
    test_stacking = pd.DataFrame(index=test_stacking_index)
    test_stacking['storage'] = test_series
    test_stacking['sarima_prediction'] = np.nan
    test_stacking['prophet_prediction'] = np.nan

    for i in range(3):
        # print(f"Starting SARIMA model training for iteration {i}")
        sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
        sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
        # print(f"\n train_series size: {train_series.shape}")
        sarima_test_pred = sarima_model.predict(steps=52)
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

    # Predicting SARIMA for test data
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series[-(52*4):])
    sarima_test_pred = sarima_model.predict(steps=52)
    sarima_test_pred.index = test_stacking.index
    sarima_test_pred.head()

    test_stacking['sarima_prediction'] = sarima_test_pred

    for i in range(3):
        # print(f"Starting Prophet model training for iteration {i}")
        prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
        prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
        # print(f"\n train_series size: {train_series.shape}")
        prophet_test_pred = prophet_model.predict(steps=52)
        # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

    # Predicting prophet for test data
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series[-(52*4):])
    # print(f"\n Selecting from {train_series.index[-(52*4):]}")
    prophet_test_pred = prophet_model.predict(steps=52)
    prophet_test_pred.index = test_stacking.index
    prophet_test_pred.head()

    test_stacking['prophet_prediction'] = prophet_test_pred
    # print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
    # test_stacking.info()

    train = pd.DataFrame(index=train_stacking.index)
    test = pd.DataFrame(index=test_stacking.index)

    train['storage'] = train_stacking['storage'].astype(float)
    train['sarima'] = train_stacking['sarima_prediction'].astype(float)
    train['prophet'] = train_stacking['prophet_prediction'].astype(float)

    test['storage'] = test_stacking['storage'].astype(float)
    test['sarima'] = test_stacking['sarima_prediction'].astype(float)
    test['prophet'] = test_stacking['prophet_prediction'].astype(float)

    # Train Linear Regression Stacking Model
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Prepare features (predictions from base models) and target (actual storage)
    X_train = train[['sarima', 'prophet']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima', 'prophet']].copy()
    y_test = test['storage'].copy()

    # Remove any rows with NaN values
    train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
    test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]

    # Make predictions
    train_stacking_pred = 0.7*X_train_clean['sarima'] + 0.3*X_train_clean['prophet']
    test_stacking_pred = 0.7*X_test_clean['sarima'] + 0.3*X_test_clean['prophet']
    print(f"\n X_test_clean: \n {X_test_clean}")
    print(f"\n test_stacking_pred: \n {test_stacking_pred}")

    # Create prediction series with proper indices
    train_stacking_predictions = pd.Series(train_stacking_pred, index=X_train_clean.index, name='stacking_prediction')
    test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

    # Calculate performance metrics
    train_mse = mean_squared_error(y_train_clean, train_stacking_pred)
    train_mae = mean_absolute_error(y_train_clean, train_stacking_pred)
    train_r2 = r2_score(y_train_clean, train_stacking_pred)

    test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
    test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
    test_r2 = r2_score(y_test_clean, test_stacking_pred)

    print("Linear Regression Stacking Model Performance:")
    print(f"Train - MSE: {train_mse:.4f}, MAE: {train_mae:.4f}, R²: {train_r2:.4f}")
    print(f"Test  - MSE: {test_mse:.4f}, MAE: {test_mae:.4f}, R²: {test_r2:.4f}")

    print(f"\nModel Coefficients:")
    print(f"SARIMA: {stacking_model.coef_[0]:.4f}")
    print(f"Prophet: {stacking_model.coef_[1]:.4f}")
    print(f"Intercept: {stacking_model.intercept_:.4f}")

    # Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train_stacking_predictions.index, train_stacking_predictions, label='Stacking Train', color='orange', linestyle='-', linewidth=2)

    # Plot test data and predictions
    plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

    # Add train/test split line
    plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Display model weights interpretation
    print(f"\nModel Interpretation:")
    total_weight = sum(abs(coef) for coef in stacking_model.coef_)
    print(f"Relative importance:")
    print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
    print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")

### More visualization comparisons

In [ ]:
ponderation_stacking(11)

In [ ]:
ponderation_stacking(1)

In [ ]:
ponderation_stacking(127)

### Second version for XGBoost, still not the definitive one

In [ ]:
# Create and train XGBoost model
model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

def predict_next_year(X):
    predictions = []
    for week in range(1, 53):
        X_step = X.copy()
        next_year_predictions = pd.concat([train_stacking, test_stacking], axis=0)
        X_step['sarima_prediction'] = next_year_predictions['sarima_prediction'].shift(-week).iloc[:156]
        X_step['prophet_prediction'] = next_year_predictions['prophet_prediction'].shift(-week).iloc[:156]
        storage_one_year_ago = X_step['storage'][-52]
        difference = storage_one_year_ago - X_step['storage'].iloc[-1]
        last_row = X_step.iloc[-1]
        X_step['next_storage_value'] = X_step['storage'].shift(-week)
        X_step = X_step.iloc[:-52]

        last_row['storage'] = last_row['storage'] + difference
        last_row['sarima_prediction'] = last_row['sarima_prediction'] + difference
        last_row['prophet_prediction'] = last_row['prophet_prediction'] + difference
        
        # print(f"\n X_step from step {week}: \n {X_step}")
        model.fit(X_step)
        
        print(f"\n Last row for week {week}: \n {last_row}")
        # Generate forecasts for 52 steps
        prediction = model.predict(last_row.to_frame().T)
        predictions.append(prediction-difference)

    # Create forecast series with same index as test_stacking
    forecast = pd.Series(predictions, index=test_stacking.index)
    return forecast

# Create comprehensive feature-rich dataset for XGBoost
# Get the reservoir data with all original features
df_res_full = df[df['id'] == reservoir_id].copy()
df_res_full = df_res_full.sort_values('date').set_index('date')

# Filter to match train_stacking index period
train_period = df_res_full.loc[train_stacking.index]

# Create the comprehensive train dataset
train = pd.DataFrame(index=train_stacking.index)

# Basic storage features
train['storage'] = train_stacking['storage'].astype(float)

'''
# Seasonal lags (quarterly and annual)
for lag in [52, 52*2]:
    train[f'storage_lag_{lag}w'] = train['storage'].shift(lag).astype(float)
'''

# Base model predictions as features
train['sarima_prediction'] = train_stacking['sarima_prediction'].astype(float)
train['prophet_prediction'] = train_stacking['prophet_prediction'].astype(float)

# Convert all columns to numeric, replacing any remaining issues with NaN
train = train.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN values with forward fill, then backward fill
train = train.fillna(method='ffill').fillna(method='bfill')

# Test the function
forecast = predict_next_year(train)


plt.figure(figsize=(12, 6))

# Plot training and test data
plt.plot(train_stacking.index, train_stacking['storage'], label='Train')
plt.plot(train_stacking.index, train_stacking['sarima_prediction'], label='SARIMA_train', color='red', linestyle='--')
plt.plot(train_stacking.index, train_stacking['prophet_prediction'], label='Prophet_train', color='green', linestyle='--')
plt.plot(test_stacking.index, test_stacking['storage'], label='Test', color='black', linewidth=2)
plt.plot(test_stacking.index, test_stacking['prophet_prediction'], label='Prophet_test')
plt.plot(test_stacking.index, forecast, label='XGBoost Stacking', color='purple')
plt.plot(test_stacking.index, test_stacking['sarima_prediction'], label='SARIMA_test', color='orange', linestyle='--')

# Add train/test split line
plt.axvline(test_stacking.index[0], color='gray', linestyle='--', label='Train/Test Split')

# Formatting
plt.legend()
title = f'Forecast for Reservoir {reservoir_id}'
plt.title(title)
plt.xlabel('Date')
plt.ylabel('Storage')
plt.tight_layout()
plt.show()

### Adding years as parameters, so that I can train models with different amount of years

In [ ]:
df_res = df[df['id'] == 11].copy()
df_res = df_res.sort_values('date')

# Set date as index
series = df_res.set_index('date')['storage']
length = len(series)
years = min(11, length // 52) - 1

# Train/test split: last 52 weeks as test
train_series = series.iloc[-(52*(years+1)):-52] # 'years' years
test_series = series.iloc[-52:] # 1 year

train_stacking_index = train_series[-(52*(years-4)):].index
# Create a dataframe with train_index as index
train_stacking = pd.DataFrame(index=train_stacking_index)
train_stacking['storage'] = train_series[-(52*(years-4)):] # 'years'-4 years, as the first 4 are for training sarima and prophet
train_stacking['sarima_prediction'] = np.nan
train_stacking['prophet_prediction'] = np.nan

test_stacking_index = test_series.index  # Last year for testing the stacking one
test_stacking = pd.DataFrame(index=test_stacking_index)
test_stacking['storage'] = test_series
test_stacking['sarima_prediction'] = np.nan
test_stacking['prophet_prediction'] = np.nan

for i in range(years-4):
    # print(f"Starting SARIMA model training for iteration {i}")
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
    # print(f"\n train_series size: {train_series.shape}")
    sarima_test_pred = sarima_model.predict(steps=52)
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

# Predicting SARIMA for test data
sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
sarima_model.fit(train_series[-(52*4):])
sarima_test_pred = sarima_model.predict(steps=52)
sarima_test_pred.index = test_stacking.index

test_stacking['sarima_prediction'] = sarima_test_pred

for i in range(years-4):
    # print(f"Starting Prophet model training for iteration {i}")
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
    # print(f"\n train_series size: {train_series.shape}")
    prophet_test_pred = prophet_model.predict(steps=52)
    # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

# Predicting prophet for test data
prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
prophet_model.fit(train_series[-(52*4):])
# print(f"\n Selecting from {train_series.index[-(52*4):]}")
prophet_test_pred = prophet_model.predict(steps=52)
prophet_test_pred.index = test_stacking.index
prophet_test_pred.head()

test_stacking['prophet_prediction'] = prophet_test_pred
# print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
# test_stacking.info()

### More sophisticated version, still not definitive

In [ ]:
def good_workflow_for_reservoir(reservoir_id, years_training):
    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    capacity = df[df['id'] == reservoir_id]['capacity'].values[0]

    # Set date as index
    series = df_res.set_index('date')['storage']
    length = len(series)
    years = min(11, length // 52) - 1

    # Train/test split: last 52 weeks as test
    train_series = series.iloc[-(52*(years+1)):-52] # 'years' years
    test_series = series.iloc[-52:] # 1 year

    train_stacking_index = train_series[-(52*(years-4)):].index
    # Create a dataframe with train_index as index
    train_stacking = pd.DataFrame(index=train_stacking_index)
    train_stacking['storage'] = train_series[-(52*(years-4)):] # 'years'-4 years, as the first 4 are for training sarima and prophet
    train_stacking['sarima_prediction'] = np.nan
    train_stacking['prophet_prediction'] = np.nan

    test_stacking_index = test_series.index  # Last year for testing the stacking one
    test_stacking = pd.DataFrame(index=test_stacking_index)
    test_stacking['storage'] = test_series
    test_stacking['sarima_prediction'] = np.nan
    test_stacking['prophet_prediction'] = np.nan

    for i in range(years-4):
        # print(f"Starting SARIMA model training for iteration {i}")
        sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
        sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
        # print(f"\n train_series size: {train_series.shape}")
        sarima_test_pred = sarima_model.predict(steps=52)
        mask = sarima_test_pred > capacity
        sarima_test_pred[mask] = capacity
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

    # Predicting SARIMA for test data
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series[-(52*4):])
    sarima_test_pred = sarima_model.predict(steps=52)
    mask = sarima_test_pred > capacity
    sarima_test_pred[mask] = capacity
    sarima_test_pred.index = test_stacking.index

    test_stacking['sarima_prediction'] = sarima_test_pred

    for i in range(years-4):
        # print(f"Starting Prophet model training for iteration {i}")
        prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
        prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
        # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
        # print(f"\n train_series size: {train_series.shape}")
        prophet_test_pred = prophet_model.predict(steps=52)
        # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
        mask = prophet_test_pred > capacity
        prophet_test_pred[mask] = capacity
        train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

    # Predicting prophet for test data
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series[-(52*4):])
    # print(f"\n Selecting from {train_series.index[-(52*4):]}")
    prophet_test_pred = prophet_model.predict(steps=52)
    mask = prophet_test_pred > capacity
    prophet_test_pred[mask] = capacity
    prophet_test_pred.index = test_stacking.index

    test_stacking['prophet_prediction'] = prophet_test_pred
    # print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
    # test_stacking.info()

    df_res = df[df['id'] == reservoir_id].copy()
    df_res = df_res.sort_values('date')

    # Set date as index
    train_series_xgboost = train_stacking.copy()
    train_stacking_xgboost = train_series_xgboost[-(52*years_training):].copy()  # Last 'years_training' years from train_series of storage
    train_stacking_xgboost.loc[:, 'storage'] = train_stacking_xgboost['storage'].astype(float)
    train_stacking_xgboost.loc[:, 'sarima_prediction'] = train_stacking_xgboost['sarima_prediction'].astype(float)
    train_stacking_xgboost.loc[:, 'prophet_prediction'] = train_stacking_xgboost['prophet_prediction'].astype(float)

    train_stacking_xgboost['xgboost_prediction'] = np.nan

    years_training_xgboost = (len(train_series_xgboost) // 52) - years_training

    test_stacking_xgboost = test_stacking.copy()
    test_stacking_xgboost.loc[:, 'storage'] = test_stacking_xgboost['storage'].astype(float)
    test_stacking_xgboost.loc[:, 'sarima_prediction'] = test_stacking_xgboost['sarima_prediction'].astype(float)
    test_stacking_xgboost.loc[:, 'prophet_prediction'] = test_stacking_xgboost['prophet_prediction'].astype(float)


    test_stacking_xgboost['xgboost_prediction'] = np.nan

    # Create and train XGBoost model
    model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

    def predict_next_year(X):
        predictions = []
        for week in range(1, 53):
            X_step = X.copy()[-(52*years_training_xgboost):]
            next_year_predictions = pd.concat([X_step, test_stacking_xgboost], axis=0)
            X_step['sarima_prediction'] = next_year_predictions['sarima_prediction'].shift(-week).iloc[:52*years_training_xgboost]
            X_step['prophet_prediction'] = next_year_predictions['prophet_prediction'].shift(-week).iloc[:52*years_training_xgboost]
            storage_one_year_ago = X_step['storage'][-52]
            difference = storage_one_year_ago - X_step['storage'].iloc[-1]
            last_row = X_step.iloc[-1]
            X_step['next_storage_value'] = X_step['storage'].shift(-week)
            X_step = X_step.iloc[:-52]

            
            last_row['storage'] = last_row['storage'] + difference
            last_row['sarima_prediction'] = last_row['sarima_prediction'] + difference
            last_row['prophet_prediction'] = last_row['prophet_prediction'] + difference
            
            
            # print(f"\n X_step from step {week}: \n {X_step}")
            model.fit(X_step)
            
            # print(f"\n Last row for week {week}: \n {last_row}")
            # Generate forecasts for 52 steps
            prediction = model.predict(last_row.to_frame().T)
            predictions.append(prediction[0]-difference)
            # predictions.append(prediction[0])

        return pd.Series(predictions, dtype=float)

    for i in range(years_training):
        xgboost_test_pred = predict_next_year(train_series_xgboost.iloc[i*52:(i+years_training_xgboost)*52])
        mask = xgboost_test_pred > capacity
        xgboost_test_pred[mask] = capacity
        train_stacking_xgboost.iloc[(52*i):(52*(i+1)), train_stacking_xgboost.columns.get_loc('xgboost_prediction')] = xgboost_test_pred.values.astype('float64')

    xgboost_test_pred = predict_next_year(train_series_xgboost.iloc[-(52*years_training_xgboost):])
    mask = xgboost_test_pred > capacity
    xgboost_test_pred[mask] = capacity
    xgboost_test_pred.index = test_stacking_xgboost.index

    test_stacking_xgboost['xgboost_prediction'] = xgboost_test_pred.astype('float64')


    train = pd.DataFrame(index=train_stacking.index)
    test = pd.DataFrame(index=test_stacking.index)

    train['storage'] = train_stacking['storage'].astype(float)
    train['sarima'] = train_stacking['sarima_prediction'].astype(float)
    train['prophet'] = train_stacking['prophet_prediction'].astype(float)
    train['xgboost'] = train_stacking_xgboost['xgboost_prediction'].astype(float)

    test['storage'] = test_stacking['storage'].astype(float)
    test['sarima'] = test_stacking['sarima_prediction'].astype(float)
    test['prophet'] = test_stacking['prophet_prediction'].astype(float)
    test['xgboost'] = test_stacking_xgboost['xgboost_prediction'].astype(float)

    # Train Linear Regression Stacking Model
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Prepare features (predictions from base models) and target (actual storage)
    X_train = train[['sarima', 'prophet', 'xgboost']].copy()
    y_train = train['storage'].copy()

    X_test = test[['sarima', 'prophet', 'xgboost']].copy()
    y_test = test['storage'].copy()

    # Remove any rows with NaN values
    train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
    test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]

    # Train the linear regression stacking model
    stacking_model = LinearRegression()
    stacking_model.fit(X_train_clean, y_train_clean)

    test_stacking_pred = stacking_model.predict(X_test_clean)

    # Create prediction series with proper indices
    test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

    test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
    test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
    test_r2 = r2_score(y_test_clean, test_stacking_pred)

    # Create comprehensive plot
    plt.figure(figsize=(15, 8))

    # Plot training data and predictions
    plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
    plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
    plt.plot(train.index, train['xgboost'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)

    # Plot test data and predictions
    plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
    plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
    plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
    plt.plot(test.index, test['xgboost'], label='XGBoost Test', color='purple', alpha=0.8)
    plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

    # Add train/test split line
    plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

    # Formatting
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
    plt.xlabel('Date')
    plt.ylabel('Storage')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Display model weights interpretation
    print(f"\nModel Interpretation:")
    total_weight = sum(abs(coef) for coef in stacking_model.coef_)
    print(f"Relative importance:")
    print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
    print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")
    print(f"XGBoost: {abs(stacking_model.coef_[2])/total_weight*100:.1f}%")

### Trying to apply Ridge instead of classical Regression

In [ ]:
# Set date as index
series = df_res.set_index('date')['storage']
length = len(series)
years = min(11, length // 52) - 1

# Train/test split: last 52 weeks as test
train_series = series.iloc[-(52*(years+1)):-52] # 'years' years
test_series = series.iloc[-52:] # 1 year

train_stacking_index = train_series[-(52*(years-4)):].index
# Create a dataframe with train_index as index
train_stacking = pd.DataFrame(index=train_stacking_index)
train_stacking['storage'] = train_series[-(52*(years-4)):] # 'years'-4 years, as the first 4 are for training sarima and prophet
train_stacking['sarima_prediction'] = np.nan
train_stacking['prophet_prediction'] = np.nan

test_stacking_index = test_series.index  # Last year for testing the stacking one
test_stacking = pd.DataFrame(index=test_stacking_index)
test_stacking['storage'] = test_series
test_stacking['sarima_prediction'] = np.nan
test_stacking['prophet_prediction'] = np.nan

for i in range(years-4):
    # print(f"Starting SARIMA model training for iteration {i}")
    sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
    sarima_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[-(52*(i+1)):-52*(i+4)]}")
    # print(f"\n train_series size: {train_series.shape}")
    sarima_test_pred = sarima_model.predict(steps=52)
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('sarima_prediction')] = sarima_test_pred

# Predicting SARIMA for test data
sarima_model = SARIMAModel(order=(1,1,1), seasonal_order=(1,1,1,52))
sarima_model.fit(train_series[-(52*4):])
sarima_test_pred = sarima_model.predict(steps=52)
sarima_test_pred.index = test_stacking.index

test_stacking['sarima_prediction'] = sarima_test_pred

for i in range(years-4):
    # print(f"Starting Prophet model training for iteration {i}")
    prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
    prophet_model.fit(train_series.iloc[(52*i):(52*(i+4))])
    # print(f"\n Selecting from {train_series.index[(52*i):(52*(i+4))]}")
    # print(f"\n train_series size: {train_series.shape}")
    prophet_test_pred = prophet_model.predict(steps=52)
    # print(f"\n Shape of Prophet predictions for iteration {i}: {prophet_test_pred.shape}")
    train_stacking.iloc[(52*i):(52*(i+1)), train_stacking.columns.get_loc('prophet_prediction')] = prophet_test_pred

# Predicting prophet for test data
prophet_model = ProphetModel(yearly_seasonality=True, changepoint_prior_scale=0.0005)
prophet_model.fit(train_series[-(52*4):])
# print(f"\n Selecting from {train_series.index[-(52*4):]}")
prophet_test_pred = prophet_model.predict(steps=52)
prophet_test_pred.index = test_stacking.index
prophet_test_pred.head()

test_stacking['prophet_prediction'] = prophet_test_pred
# print(f"\nFinal train stacking DataFrame shape: {train_stacking.shape}")
# test_stacking.info()

df_res = df[df['id'] == reservoir_id].copy()
df_res = df_res.sort_values('date')

# Set date as index
train_series_xgboost = train_stacking.copy()
train_stacking_xgboost = train_series_xgboost[-(52*years_training):].copy()  # Last 'years_training' years from train_series of storage
train_stacking_xgboost.loc[:, 'storage'] = train_stacking_xgboost['storage'].astype(float)
train_stacking_xgboost.loc[:, 'sarima_prediction'] = train_stacking_xgboost['sarima_prediction'].astype(float)
train_stacking_xgboost.loc[:, 'prophet_prediction'] = train_stacking_xgboost['prophet_prediction'].astype(float)

train_stacking_xgboost['xgboost_prediction'] = np.nan

years_training_xgboost = (len(train_series_xgboost) // 52) - years_training

test_stacking_xgboost = test_stacking.copy()
test_stacking_xgboost.loc[:, 'storage'] = test_stacking_xgboost['storage'].astype(float)
test_stacking_xgboost.loc[:, 'sarima_prediction'] = test_stacking_xgboost['sarima_prediction'].astype(float)
test_stacking_xgboost.loc[:, 'prophet_prediction'] = test_stacking_xgboost['prophet_prediction'].astype(float)


test_stacking_xgboost['xgboost_prediction'] = np.nan

# Create and train XGBoost model
model = XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1)

def predict_next_year(X):
    predictions = []
    for week in range(1, 53):
        X_step = X.copy()[-(52*years_training_xgboost):]
        next_year_predictions = pd.concat([X_step, test_stacking_xgboost], axis=0)
        X_step['sarima_prediction'] = next_year_predictions['sarima_prediction'].shift(-week).iloc[:52*years_training_xgboost]
        X_step['prophet_prediction'] = next_year_predictions['prophet_prediction'].shift(-week).iloc[:52*years_training_xgboost]
        storage_one_year_ago = X_step['storage'][-52]
        difference = storage_one_year_ago - X_step['storage'].iloc[-1]
        last_row = X_step.iloc[-1]
        X_step['next_storage_value'] = X_step['storage'].shift(-week)
        X_step = X_step.iloc[:-52]

        
        last_row['storage'] = last_row['storage'] + difference
        last_row['sarima_prediction'] = last_row['sarima_prediction'] + difference
        last_row['prophet_prediction'] = last_row['prophet_prediction'] + difference
        
        
        # print(f"\n X_step from step {week}: \n {X_step}")
        model.fit(X_step)
        
        # print(f"\n Last row for week {week}: \n {last_row}")
        # Generate forecasts for 52 steps
        prediction = model.predict(last_row.to_frame().T)
        predictions.append(prediction[0]-difference)
        # predictions.append(prediction[0])

    return pd.Series(predictions, dtype=float)

for i in range(years_training):
    xgboost_test_pred = predict_next_year(train_series_xgboost.iloc[i*52:(i+years_training_xgboost)*52])
    # FIX: Convert to proper dtype before assignment
    train_stacking_xgboost.iloc[(52*i):(52*(i+1)), train_stacking_xgboost.columns.get_loc('xgboost_prediction')] = xgboost_test_pred.values.astype('float64')

xgboost_test_pred = predict_next_year(train_series_xgboost.iloc[-(52*years_training_xgboost):])
xgboost_test_pred.index = test_stacking_xgboost.index

test_stacking_xgboost['xgboost_prediction'] = xgboost_test_pred.astype('float64')


train = pd.DataFrame(index=train_stacking.index)
test = pd.DataFrame(index=test_stacking.index)

train['storage'] = train_stacking['storage'].astype(float)
train['sarima'] = train_stacking['sarima_prediction'].astype(float)
train['prophet'] = train_stacking['prophet_prediction'].astype(float)
train['xgboost'] = train_stacking_xgboost['xgboost_prediction'].astype(float)

test['storage'] = test_stacking['storage'].astype(float)
test['sarima'] = test_stacking['sarima_prediction'].astype(float)
test['prophet'] = test_stacking['prophet_prediction'].astype(float)
test['xgboost'] = test_stacking_xgboost['xgboost_prediction'].astype(float)

# Train Ridge Regression Stacking Model
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare features (predictions from base models) and target (actual storage)
X_train = train[['sarima', 'prophet', 'xgboost']].copy()
y_train = train['storage'].copy()

X_test = test[['sarima', 'prophet', 'xgboost']].copy()
y_test = test['storage'].copy()

# Remove any rows with NaN values
train_mask = ~(X_train.isna().any(axis=1) | y_train.isna())
test_mask = ~(X_test.isna().any(axis=1) | y_test.isna())

X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]
X_test_clean = X_test[test_mask]
y_test_clean = y_test[test_mask]

# Train the Ridge regression stacking model (regularized linear)
stacking_model = Ridge(alpha=1.0)  # tune alpha as needed
stacking_model.fit(X_train_clean, y_train_clean)

test_stacking_pred = stacking_model.predict(X_test_clean)

# Create prediction series with proper indices
test_stacking_predictions = pd.Series(test_stacking_pred, index=X_test_clean.index, name='stacking_prediction')

test_mse = mean_squared_error(y_test_clean, test_stacking_pred)
test_mae = mean_absolute_error(y_test_clean, test_stacking_pred)
test_r2 = r2_score(y_test_clean, test_stacking_pred)

# Create comprehensive plot
plt.figure(figsize=(15, 8))

# Plot training data and predictions
plt.plot(train.index, train['storage'], label='Train Actual', color='blue', linewidth=2)
plt.plot(train.index, train['sarima'], label='SARIMA Train', color='red', linestyle='--', alpha=0.7)
plt.plot(train.index, train['prophet'], label='Prophet Train', color='green', linestyle='--', alpha=0.7)
plt.plot(train.index, train['xgboost'], label='XGBoost Train', color='purple', linestyle='--', alpha=0.7)

# Plot test data and predictions
plt.plot(test.index, test['storage'], label='Test Actual', color='black', linewidth=3)
plt.plot(test.index, test['sarima'], label='SARIMA Test', color='red', alpha=0.8)
plt.plot(test.index, test['prophet'], label='Prophet Test', color='green', alpha=0.8)
plt.plot(test.index, test['xgboost'], label='XGBoost Test', color='purple', alpha=0.8)
plt.plot(test_stacking_predictions.index, test_stacking_predictions, label='Stacking Test', color='orange', linewidth=2)

# Add train/test split line
plt.axvline(test.index[0], color='gray', linestyle='--', alpha=0.8, label='Train/Test Split')

# Formatting
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title(f'Stacking Ensemble Forecast for Reservoir {reservoir_id}')
plt.xlabel('Date')
plt.ylabel('Storage')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Display model weights interpretation
print(f"\nModel Interpretation:")
total_weight = sum(abs(coef) for coef in stacking_model.coef_)
print(f"Relative importance:")
print(f"SARIMA: {abs(stacking_model.coef_[0])/total_weight*100:.1f}%")
print(f"Prophet: {abs(stacking_model.coef_[1])/total_weight*100:.1f}%")
print(f"XGBoost: {abs(stacking_model.coef_[2])/total_weight*100:.1f}%")

### Slight contact with LSTMs

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibility
np.random.seed(27)
tf.random.set_seed(27)

# Configuration
SEQ_LEN = 12        # lookback (weeks)
EPOCHS = 200
BATCH_SIZE = 16
PATIENCE = 12

# Helper: create sequences (X: seq_len x n_features, y: scalar next storage)
def create_sequences(df_features, seq_len=SEQ_LEN):
    values = df_features[['storage', 'sarima', 'prophet', 'xgboost']].values.astype(float)
    X, y = [], []
    for i in range(seq_len, len(values)):
        X.append(values[i-seq_len:i, :])   # window ends at i-1
        y.append(values[i, 0])             # storage at time i
    return np.array(X), np.array(y)

# Build LSTM model
def build_lstm(input_shape):
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))
    model.add(layers.LSTM(64, activation='tanh'))
    model.add(layers.Dropout(0.2))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

# Forecast multi-step using iterative predictions and future exogenous DataFrame
def forecast_multi_step(model, last_window, exog_future_df, scaler_X, scaler_y, steps):
    # last_window: raw (unscaled) array shape (seq_len, n_features) with columns: storage,sarima,prophet,xgboost
    preds = []
    window = last_window.copy()
    for step in range(steps):
        # scale window
        scaled_window = scaler_X.transform(window.reshape(-1, window.shape[-1])).reshape(1, window.shape[0], window.shape[1])
        y_scaled = model.predict(scaled_window, verbose=0)
        # inverse transform predicted storage
        y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1,1)).flatten()[0]
        preds.append(float(y_pred))
        # prepare next row to append to window: storage = predicted, exog = exog_future_df.iloc[step]
        exog_vals = exog_future_df.iloc[step][['sarima','prophet','xgboost']].values.astype(float)
        next_row = np.concatenate([[y_pred], exog_vals])
        # slide window
        window = np.vstack([window[1:], next_row])
    return np.array(preds)

# === Prepare training data (uses existing `train` and `test` DataFrames in the notebook) ===
# Expectation: `train` and `test` DataFrames exist with columns: 'storage','sarima','prophet','xgboost'

if 'train' not in globals() or 'test' not in globals():
    raise RuntimeError("`train` and `test` DataFrames must exist in the notebook (with columns storage,sarima,prophet,xgboost)")

features_df = pd.concat([train, test], axis=0)

# Fit scalers on training portion only
train_features_df = train[['storage','sarima','prophet','xgboost']].copy()

scaler_X = MinMaxScaler()
# scaler_X is fit on full feature space (each column)
scaler_X.fit(train_features_df.values)

# We'll scale target (storage) separately
scaler_y = MinMaxScaler()
scaler_y.fit(train_features_df[['storage']].values)

# Create sequences from the combined 'train' frame only (we predict test later)
X_all, y_all = create_sequences(train_features_df, seq_len=SEQ_LEN)

# Scale X sequences (reshape to 2D then back)
n_samples, seq_len, n_features = X_all.shape
X_all_2d = X_all.reshape(-1, n_features)
X_all_2d_scaled = scaler_X.transform(X_all_2d)
X_all_scaled = X_all_2d_scaled.reshape(n_samples, seq_len, n_features)

# Scale y
y_all_scaled = scaler_y.transform(y_all.reshape(-1,1)).flatten()

# Train/validation split (last 10% as val)
split_idx = int(len(X_all_scaled) * 0.9)
X_train_seq, X_val_seq = X_all_scaled[:split_idx], X_all_scaled[split_idx:]
y_train_seq, y_val_seq = y_all_scaled[:split_idx], y_all_scaled[split_idx:]

# Build and train model
model = build_lstm(input_shape=(SEQ_LEN, n_features))
early = callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early],
    verbose=2
)

# Plot training history
plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(); plt.title('LSTM training')
plt.show()

# Evaluate on training (OOF) portion
train_pred_scaled = model.predict(X_all_scaled)
train_pred = scaler_y.inverse_transform(train_pred_scaled).flatten()
train_y = y_all
print('OOF MAE:', mean_absolute_error(train_y, train_pred), 'RMSE:', np.sqrt(mean_squared_error(train_y, train_pred)))

# === Multi-step forecasting on `test` using iterative predictions ===
# Build last window from the last SEQ_LEN rows of the features sequence (raw values)
full_features = pd.concat([train, test], axis=0)[['storage','sarima','prophet','xgboost']]
last_window_raw = full_features.iloc[len(train)-SEQ_LEN:len(train)].values.astype(float)

# exogenous values for the forecast horizon (test portion)
exog_future = test[['sarima','prophet','xgboost']].copy().reset_index(drop=True)

steps = len(test)
preds = forecast_multi_step(model, last_window_raw, exog_future, scaler_X, scaler_y, steps)

# Create Series aligned with test index
pred_series = pd.Series(preds, index=test.index, name='lstm_prediction')

# Metrics
print('Test MAE:', mean_absolute_error(test['storage'].values, pred_series.values),
      'Test RMSE:', np.sqrt(mean_squared_error(test['storage'].values, pred_series.values)))

# Plot results
plt.figure(figsize=(12,6))
plt.plot(train.index, train['storage'], label='Train Actual')
plt.plot(test.index, test['storage'], label='Test Actual', linewidth=2)
plt.plot(pred_series.index, pred_series.values, label='LSTM Forecast', color='orange')
plt.legend(); plt.title('LSTM multi-step forecast')
plt.show()



In [ ]:
# LSTM trained to predict `next_storage_value` using past sequences of ['storage','sarima','prophet','xgboost']
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEQ_LEN = 12
EPOCHS = 100
BATCH_SIZE = 16
PATIENCE = 10

# Expect `train` and `test` DataFrames to exist with columns: storage,sarima,prophet,xgboost,next_storage_value
for name in ('train','test'):
    if name not in globals():
        raise RuntimeError(f"DataFrame '{name}' not found in notebook namespace")

train_df = train.copy()
test_df = test.copy()
required_cols = ['storage','sarima','prophet','xgboost','next_storage_value']
for c in required_cols:
    if c not in train_df.columns or c not in test_df.columns:
        raise RuntimeError(f"Column '{c}' missing from train/test DataFrames")

# Combine for easy indexing
combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
features = ['storage','sarima','prophet','xgboost']
combined_vals = combined[features].astype(float).values

# Build training sequences from train_df portion
n_train = len(train_df)
X_train, y_train = [], []
for i in range(SEQ_LEN, n_train):
    X_train.append(combined_vals[i-SEQ_LEN:i, :])
    y_train.append(combined.loc[i, 'next_storage_value'])
X_train = np.array(X_train)
y_train = np.array(y_train)

# Remove samples with NaN target
valid_mask = ~np.isnan(y_train)
X_train = X_train[valid_mask]
y_train = y_train[valid_mask]

# Scale features and target
scaler_X = MinMaxScaler()
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
scaler_X.fit(X_train_2d)
X_train_scaled = scaler_X.transform(X_train_2d).reshape(X_train.shape)

scaler_y = MinMaxScaler()
scaler_y.fit(y_train.reshape(-1,1))
y_train_scaled = scaler_y.transform(y_train.reshape(-1,1)).flatten()

# Train/validation split
split = int(0.9 * len(X_train_scaled))
X_tr, X_val = X_train_scaled[:split], X_train_scaled[split:]
y_tr, y_val = y_train_scaled[:split], y_train_scaled[split:]

# Build model
def build_model(input_shape):
    m = models.Sequential()
    m.add(layers.Input(shape=input_shape))
    m.add(layers.LSTM(32, activation='tanh'))
    m.add(layers.Dropout(0.2))
    m.add(layers.Dense(16, activation='relu'))
    m.add(layers.Dense(1))
    m.compile(optimizer='adam', loss='mse')
    return m

model = build_model((SEQ_LEN, len(features)))
early = callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)

history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early], verbose=2)

# Prepare test sequences by building windows ending at each test row index
n_test = len(test_df)
start_idx = n_train
X_test_windows = []
valid_test_idx = []
for t in range(n_test):
    pos = start_idx + t
    if pos - SEQ_LEN < 0:
        # not enough history to build sequence
        continue
    window = combined_vals[pos-SEQ_LEN:pos, :]
    X_test_windows.append(window)
    valid_test_idx.append(t)

if len(X_test_windows) == 0:
    raise RuntimeError('No test windows could be built; increase training length or reduce SEQ_LEN')

X_test_windows = np.array(X_test_windows)
X_test_2d = X_test_windows.reshape(-1, X_test_windows.shape[-1])
X_test_scaled = scaler_X.transform(X_test_2d).reshape(X_test_windows.shape)

# Predict and inverse-scale
y_test_pred_scaled = model.predict(X_test_scaled)
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled).flatten()

# Align predictions with test index (only for windows we could build)
pred_series = pd.Series(index=test_df.index[valid_test_idx], data=y_test_pred)

# True test targets (for same valid indices)
y_test_true = test_df['next_storage_value'].iloc[valid_test_idx].astype(float).values

# Compute metrics (drop NaNs)
mask = ~np.isnan(y_test_true)
if mask.sum() == 0:
    print('No valid test targets to evaluate')
else:
    mae = mean_absolute_error(y_test_true[mask], y_test_pred[mask])
    rmse = np.sqrt(mean_squared_error(y_test_true[mask], y_test_pred[mask]))
    print(f'LSTM Test MAE: {mae:.4f}, RMSE: {rmse:.4f}')

# Plot
plt.figure(figsize=(12,6))
plt.plot(train_df.index, train_df['next_storage_value'], label='Train (target)', alpha=0.7)
plt.plot(test_df.index, test_df['next_storage_value'], label='Test (target)', linewidth=2, color='black')
plt.plot(pred_series.index, pred_series.values, label='LSTM prediction', color='orange')
plt.legend(); plt.title('LSTM predicting next_storage_value'); plt.show()
